# Miniforge — Test & Benchmark Notebook

**MiniMax M2.7 GGUF inference library for GMKtech M7**

This notebook:
1. Mounts Google Drive for output storage
2. Installs Miniforge and its dependencies
3. Runs the core test suite
4. Runs benchmark simulations and real hardware probes
5. Generates graphs for every major metric
6. Saves all figures + a JSON report to Drive


## 1  Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/miniforge_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

## 2  Install Dependencies

In [ ]:
# Clone the repo and install in editable mode
!git clone https://github.com/Jackson57279/miniforge.git /content/miniforge 2>/dev/null || \
    (cd /content/miniforge && git pull)

# Install core deps (skip llama-cpp-python heavy build in Colab)
!pip install -q \
    transformers>=4.40.0 \
    psutil>=5.9 \
    pydantic>=2.0 \
    pyyaml>=6.0 \
    tqdm>=4.66 \
    numpy>=1.24 \
    matplotlib>=3.8 \
    seaborn>=0.13 \
    plotly>=5.20 \
    kaleido  # plotly static export

# Install miniforge itself without heavy optional deps
!pip install -q -e /content/miniforge --no-deps

print('Installation complete.')

## 3  Imports & Helpers

In [ ]:
import sys
sys.path.insert(0, '/content/miniforge/src')

import json
import time
import psutil
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime

# ── style ──────────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.15)
PALETTE = sns.color_palette('muted')
matplotlib.rcParams.update({'figure.dpi': 150, 'savefig.bbox': 'tight'})

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

def save(fig, name: str):
    """Save a matplotlib figure to Drive and display it."""
    path = f'{OUTPUT_DIR}/{name}_{TIMESTAMP}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved → {path}')
    plt.show()

def save_plotly(fig, name: str):
    """Save a Plotly figure to Drive as PNG and HTML."""
    png_path = f'{OUTPUT_DIR}/{name}_{TIMESTAMP}.png'
    html_path = f'{OUTPUT_DIR}/{name}_{TIMESTAMP}.html'
    fig.write_image(png_path)
    fig.write_html(html_path)
    print(f'  Saved → {png_path}')
    fig.show()

print('Imports OK.')

## 4  Core Unit Tests

In [ ]:
# Run pytest and capture output
!cd /content/miniforge && pip install -q pytest pytest-asyncio && \
    python -m pytest tests/ -v --tb=short 2>&1 | tee /tmp/pytest_output.txt

# Parse results for visualisation
with open('/tmp/pytest_output.txt') as f:
    pytest_output = f.read()

passed = pytest_output.count(' PASSED')
failed = pytest_output.count(' FAILED')
errors = pytest_output.count(' ERROR')
skipped = pytest_output.count(' SKIPPED')

print(f'\nTest summary: {passed} passed, {failed} failed, {errors} errors, {skipped} skipped')

In [ ]:
# ── Figure 1: Test results donut ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))

labels, sizes, colors = [], [], []
if passed:  labels.append('Passed');  sizes.append(passed);  colors.append('#4caf50')
if failed:  labels.append('Failed');  sizes.append(failed);  colors.append('#f44336')
if errors:  labels.append('Errors');  sizes.append(errors);  colors.append('#ff9800')
if skipped: labels.append('Skipped'); sizes.append(skipped); colors.append('#9e9e9e')

if not sizes:
    labels, sizes, colors = ['No tests found'], [1], ['#9e9e9e']

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(width=0.55)
)
for at in autotexts:
    at.set_fontsize(13)

ax.set_title('Unit Test Results', fontsize=16, fontweight='bold', pad=20)
total = sum(sizes)
ax.text(0, 0, f'{total}\ntotal', ha='center', va='center', fontsize=14, fontweight='bold')

save(fig, 'fig01_test_results')

## 5  Memory Manager Analysis

In [ ]:
from miniforge.core.memory import MemoryManager

mem = MemoryManager()
stats = mem.get_stats()

print(f'System RAM      : {stats.total_gb:.1f} GB')
print(f'Available       : {stats.available_gb:.1f} GB')
print(f'Used            : {stats.used_gb:.1f} GB ({stats.percent_used:.1f}%)')
print(f'Max usable      : {mem.max_usable_gb:.1f} GB (safety factor applied)')

In [ ]:
# ── Figure 2: RAM breakdown stacked bar ────────────────────────────────────
stats = mem.get_stats()

# Colab doesn't have 28 GB — show ACTUAL system memory
categories = ['System RAM']
os_reserve = max(stats.total_gb * 0.15, 2.0)
model_mem  = stats.total_gb * 0.55   # illustrative allocation
kv_mem     = stats.total_gb * 0.15
free_mem   = stats.total_gb - os_reserve - model_mem - kv_mem
free_mem   = max(free_mem, 0)

fig, ax = plt.subplots(figsize=(8, 4))
bar_kw = dict(height=0.5)

bottoms = [0]
for val, label, color in [
    (os_reserve, f'OS Reserve ({os_reserve:.1f} GB)', '#ef5350'),
    (model_mem,  f'Model Weights ({model_mem:.1f} GB)', '#42a5f5'),
    (kv_mem,     f'KV Cache ({kv_mem:.1f} GB)', '#66bb6a'),
    (free_mem,   f'Free ({free_mem:.1f} GB)', '#bdbdbd'),
]:
    ax.barh(categories, [val], left=bottoms, color=color, label=label, **bar_kw)
    bottoms = [bottoms[0] + val]

ax.set_xlabel('Memory (GB)', fontsize=12)
ax.set_title(f'RAM Allocation — {stats.total_gb:.1f} GB Total', fontsize=14, fontweight='bold')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.45), ncol=2)
ax.set_xlim(0, stats.total_gb * 1.05)

save(fig, 'fig02_ram_allocation')

## 6  Quantization Size Comparison

In [ ]:
# ── Figure 3: Quantization sizes for 228B MoE model ───────────────────────
model_fp16_gb = 228 * 2  # FP16: 2 bytes/param
quants = {
    'FP16':      1.000,
    'Q8_0':      0.500,
    'Q6_K':      0.375,
    'Q5_K_M':    0.313,
    'Q4_K_M':    0.250,
    'Q3_K_M':    0.188,
    'IQ2_XXS':   0.134,   # ~61 GB on disk for 228B MoE
    'UD-TQ1_0':  0.083,   # ~38 GB on disk
}

labels = list(quants.keys())
sizes  = [model_fp16_gb * r for r in quants.values()]
# Highlight which fit in 24 GB budget (M7 max usable)
target_gb = 24
bar_colors = ['#4caf50' if s <= target_gb else '#ef5350' for s in sizes]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, sizes, color=bar_colors, edgecolor='white', linewidth=1.2)
ax.axhline(target_gb, color='orange', linestyle='--', linewidth=2, label=f'{target_gb} GB RAM budget (M7)')

for bar, sz in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{sz:.0f} GB', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Disk / RAM size (GB)', fontsize=12)
ax.set_title('228B MoE Model Size by Quantization', fontsize=14, fontweight='bold')
ax.legend()
green_patch = mpatches.Patch(color='#4caf50', label='Fits in 24 GB')
red_patch   = mpatches.Patch(color='#ef5350', label='Exceeds RAM')
ax.legend(handles=[green_patch, red_patch,
                   plt.Line2D([0], [0], color='orange', linestyle='--', linewidth=2,
                              label=f'{target_gb} GB budget')])

save(fig, 'fig03_quantization_sizes')

## 7  Context Window vs Memory

In [ ]:
# ── Figure 4: Context window vs KV-cache memory ───────────────────────────
kv_types = {
    'f16':    64,   # KB/token
    'q8_0':   32,
    'q4_0':   16,
    'turbo4': 18,
    'turbo3': 14,
}
context_tokens = np.array([1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 194560])
context_k = context_tokens / 1000  # in thousands

fig, ax = plt.subplots(figsize=(11, 5))

for (kv_name, kb_per_tok), color in zip(kv_types.items(), sns.color_palette('tab10')):
    gb = context_tokens * kb_per_tok * 1024 / (1024 ** 3)
    ax.plot(context_tokens / 1000, gb, marker='o', label=kv_name, color=color, linewidth=2)

# Budget line
model_size_gb = 3.1  # approximate for Q4_K_M 2.7B equiv
budget = mem.max_usable_gb - model_size_gb - 2.0
ax.axhline(budget, color='red', linestyle='--', linewidth=2, label=f'Usable budget ({budget:.1f} GB)')

ax.set_xlabel('Context Window (K tokens)', fontsize=12)
ax.set_ylabel('KV Cache Memory (GB)', fontsize=12)
ax.set_title('KV Cache Memory vs Context Window by Quantization Type', fontsize=14, fontweight='bold')
ax.legend(title='KV type', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticks(context_tokens / 1000)
ax.set_xticklabels([f'{int(c)}K' if c >= 1 else f'{int(c*1000)}' for c in context_tokens / 1000],
                   rotation=30, ha='right')

save(fig, 'fig04_context_vs_memory')

## 8  Auto-Quantization Selection Heatmap

In [ ]:
# ── Figure 5: quantization selected per model size × memory budget ─────────
model_sizes = [1.3, 2.7, 7.0, 13.0, 30.0, 70.0]
budgets_gb  = [8, 12, 16, 20, 24, 32]

quant_order = ['Q2_K', 'Q3_K_M', 'Q4_K_M', 'Q5_K_M', 'Q6_K', 'Q8_0']
quant_to_idx = {q: i for i, q in enumerate(quant_order)}

grid = np.zeros((len(model_sizes), len(budgets_gb)), dtype=float)
annot = np.empty_like(grid, dtype=object)

for r, params in enumerate(model_sizes):
    for c, budget in enumerate(budgets_gb):
        mgr = MemoryManager(target_utilization=budget / (budget + 4.0))
        q = mgr.select_quantization(params)
        grid[r, c] = quant_to_idx.get(q, 0)
        annot[r, c] = q

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap('RdYlGn', len(quant_order))
im = ax.imshow(grid, cmap=cmap, vmin=-0.5, vmax=len(quant_order) - 0.5, aspect='auto')

for r in range(len(model_sizes)):
    for c in range(len(budgets_gb)):
        ax.text(c, r, annot[r, c], ha='center', va='center', fontsize=9,
                color='black', fontweight='bold')

ax.set_xticks(range(len(budgets_gb)))
ax.set_xticklabels([f'{b} GB' for b in budgets_gb])
ax.set_yticks(range(len(model_sizes)))
ax.set_yticklabels([f'{p}B' for p in model_sizes])
ax.set_xlabel('Available RAM Budget', fontsize=12)
ax.set_ylabel('Model Parameters', fontsize=12)
ax.set_title('Auto-Selected Quantization by Model Size & RAM Budget', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, ticks=range(len(quant_order)))
cbar.ax.set_yticklabels(quant_order)
cbar.set_label('Quantization Quality →', rotation=270, labelpad=15)

save(fig, 'fig05_quantization_heatmap')

## 9  Throughput Simulation

In [ ]:
# ── Synthetic throughput benchmark ─────────────────────────────────────────
# These are representative values from the benchmark suite definitions.
# Run with a real model to get actual measurements.

import random
random.seed(42)
np.random.seed(42)

def simulate_throughput(base_tps: float, n: int = 10, noise: float = 0.08) -> list[float]:
    """Simulate repeated throughput measurements around a base value."""
    return [base_tps * (1 + np.random.normal(0, noise)) for _ in range(n)]

prompt_types  = ['short_qa', 'code_gen', 'summarization', 'long_context']
# Rough tok/s for Q4_K_M 2.7B on Ryzen 7 PRO 6850H, 8 threads
base_tps      = [42.0, 35.0, 38.0, 18.0]

samples = {pt: simulate_throughput(bps) for pt, bps in zip(prompt_types, base_tps)}

# ── Figure 6: Box + strip plot of throughput ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

data_flat  = [v for vals in samples.values() for v in vals]
label_flat = [k for k, vals in samples.items() for _ in vals]

sns.boxplot(x=label_flat, y=data_flat, ax=ax, palette='muted',
            width=0.4, flierprops=dict(marker='o', alpha=0.4))
sns.stripplot(x=label_flat, y=data_flat, ax=ax, color='black',
              size=4, alpha=0.5, jitter=True)

ax.set_xlabel('Prompt Type', fontsize=12)
ax.set_ylabel('Throughput (tokens / second)', fontsize=12)
ax.set_title('Inference Throughput by Prompt Type\n(simulated — replace with real model run)',
             fontsize=13, fontweight='bold')

means = {pt: np.mean(vals) for pt, vals in samples.items()}
for i, (pt, mean) in enumerate(means.items()):
    ax.text(i, mean + 0.8, f'{mean:.1f}', ha='center', fontsize=10, color='navy')

save(fig, 'fig06_throughput_boxplot')

In [ ]:
# ── Figure 7: Throughput vs context window (interactive Plotly) ────────────
ctx_sizes = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 194560]
# Throughput drops as context grows (prefill cost)
tps_trend = [48, 46, 43, 39, 33, 25, 17, 10, 7]
ttft_ms   = [120, 150, 210, 340, 600, 1100, 2200, 4500, 7000]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Generation Throughput vs Context', 'Time-to-First-Token vs Context')
)

fig.add_trace(
    go.Scatter(x=ctx_sizes, y=tps_trend, mode='lines+markers',
               name='tok/s', line=dict(color='royalblue', width=3),
               marker=dict(size=8)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=ctx_sizes, y=ttft_ms, mode='lines+markers',
               name='TTFT (ms)', line=dict(color='tomato', width=3),
               marker=dict(size=8)),
    row=1, col=2
)

fig.update_xaxes(title_text='Context Window (tokens)', type='log', row=1, col=1)
fig.update_xaxes(title_text='Context Window (tokens)', type='log', row=1, col=2)
fig.update_yaxes(title_text='Tokens / second', row=1, col=1)
fig.update_yaxes(title_text='TTFT (ms)', row=1, col=2)

fig.update_layout(
    title='Miniforge Performance vs Context Window (simulated)',
    height=450, template='plotly_dark', showlegend=True
)

save_plotly(fig, 'fig07_throughput_vs_context')

## 10  Memory Scaling

In [ ]:
# ── Figure 8: Real memory sampling during token generation sim ─────────────
import threading

mem_snapshots = []

def sample_memory(stop_event, interval=0.05):
    proc = psutil.Process()
    while not stop_event.is_set():
        mem_snapshots.append(proc.memory_info().rss / (1024 ** 2))  # MB
        time.sleep(interval)

stop_event = threading.Event()
t = threading.Thread(target=sample_memory, args=(stop_event,), daemon=True)
t.start()

# Simulate work: allocate and process arrays as a proxy for model inference
arrays = []
for tokens in [1024, 4096, 16384, 65536]:
    arr = np.random.randn(tokens, 128).astype(np.float32)
    _ = np.dot(arr, arr.T[:128, :]).mean()  # force computation
    arrays.append(arr)
    time.sleep(0.2)
arrays.clear()
import gc; gc.collect()
time.sleep(0.3)

stop_event.set()
t.join()

timestamps = np.linspace(0, len(mem_snapshots) * 0.05, len(mem_snapshots))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timestamps, mem_snapshots, color='steelblue', linewidth=1.5)
ax.fill_between(timestamps, mem_snapshots, alpha=0.2, color='steelblue')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('RSS Memory (MB)', fontsize=12)
ax.set_title('Process Memory During Simulated Inference Workload', fontsize=13, fontweight='bold')
ax.axhline(np.mean(mem_snapshots), color='red', linestyle='--', alpha=0.7,
           label=f'Mean: {np.mean(mem_snapshots):.0f} MB')
ax.legend()

save(fig, 'fig08_memory_over_time')

## 11  Quality Metrics Radar

In [ ]:
# ── Figure 9: Quality metrics radar chart (Plotly) ─────────────────────────
# Simulated quality scores — replace with actual benchmark_suite.py output
categories = ['QA Accuracy', 'Reasoning', 'Summarization',
               'Instruction\nFollowing', 'Coherence', 'Tool Calling']

scores_q4km   = [0.82, 0.75, 0.88, 0.91, 0.85, 0.79]
scores_q3km   = [0.76, 0.68, 0.83, 0.87, 0.80, 0.71]
scores_iq2xxs = [0.68, 0.60, 0.76, 0.81, 0.73, 0.62]

cats_closed = categories + [categories[0]]   # close the polygon

fig = go.Figure()
for scores, name, color in [
    (scores_q4km,   'Q4_K_M',   'royalblue'),
    (scores_q3km,   'Q3_K_M',   'tomato'),
    (scores_iq2xxs, 'IQ2_XXS',  'mediumseagreen'),
]:
    fig.add_trace(go.Scatterpolar(
        r=scores + [scores[0]],
        theta=cats_closed,
        fill='toself',
        name=name,
        line_color=color,
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Quality Metrics by Quantization (simulated)',
    template='plotly_dark',
    height=520
)

save_plotly(fig, 'fig09_quality_radar')

## 12  Context Memory Scaling (AirLLM Method)

In [ ]:
# ── Figure 10: compute_moe_context across RAM configs ─────────────────────
from unittest.mock import patch

ram_configs_gb = [8, 12, 16, 20, 24, 28, 32, 48, 64]
kv_types_eval  = ['q4_0', 'q8_0', 'f16']

results = {kv: [] for kv in kv_types_eval}

for ram in ram_configs_gb:
    # Patch psutil to simulate different RAM amounts
    mock_vm = psutil.virtual_memory()._asdict()
    mock_vm['total'] = int(ram * 1024 ** 3)
    mock_vm['available'] = int(ram * 0.6 * 1024 ** 3)

    with patch('psutil.virtual_memory', return_value=psutil.virtual_memory().__class__(**mock_vm)):
        for kv in kv_types_eval:
            ctx = mem.compute_moe_context(
                model_disk_gb=61,   # IQ2_XXS on disk
                n_layers=62,
                n_kv_heads=8,
                head_dim=128,
                kv_cache_type=kv,
                is_moe=True
            )
            results[kv].append(ctx)

fig, ax = plt.subplots(figsize=(11, 5))

for kv, color in zip(kv_types_eval, ['#42a5f5', '#66bb6a', '#ef5350']):
    ax.plot(ram_configs_gb, results[kv], marker='o', linewidth=2,
            label=f'KV={kv}', color=color)

# Mark the M7 target
ax.axvline(28, color='orange', linestyle=':', linewidth=2, label='GMKtech M7 (28 GB)')

ax.set_xlabel('System RAM (GB)', fontsize=12)
ax.set_ylabel('Max Safe n_ctx (tokens)', fontsize=12)
ax.set_title('AirLLM Context Sizing — Max n_ctx for 228B MoE (61 GB GGUF)', fontsize=13, fontweight='bold')
ax.legend(title='KV cache type')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

save(fig, 'fig10_airllm_context_sizing')

## 13  Optimal Settings Matrix

In [ ]:
# ── Figure 11: Optimal settings table rendered as heatmap ─────────────────
from miniforge.utils.config import M7Config

model_params = [1.3, 2.7, 7.0, 13.0]
all_settings = []
for params in model_params:
    s = mem.get_optimal_settings(params)
    all_settings.append(s)

# Build a tidy summary table
import pandas as pd

df = pd.DataFrame(all_settings, index=[f'{p}B' for p in model_params])
df = df[['quantization', 'n_ctx', 'cache_type_k', 'n_threads', 'n_batch', 'flash_attn']]
df.index.name = 'Model'

print(df.to_string())

# Render as a styled table figure
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
tbl = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    rowLabels=df.index,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 2.0)

# Header styling
for (row, col), cell in tbl.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#37474f')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#eceff1')

ax.set_title('Auto-Derived Optimal Settings per Model Size', fontsize=13,
             fontweight='bold', pad=20)

save(fig, 'fig11_optimal_settings_table')

## 14  CPU Utilisation & Batch Size Impact

In [ ]:
# ── Figure 12: Throughput vs n_threads (simulated curve) ──────────────────
threads  = [1, 2, 4, 6, 8, 10, 12, 16]
# Amdahl's law: f_parallel = 0.92
f = 0.92
base_tps_single = 6.0
speedup = [1 / ((1 - f) + f / t) for t in threads]
tps_est = [base_tps_single * s for s in speedup]

# Add noise
tps_noisy = [v * (1 + np.random.normal(0, 0.04)) for v in tps_est]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(threads, tps_est,   color='steelblue', linewidth=2, label='Amdahl estimate (f=0.92)')
ax.plot(threads, tps_noisy, 'o', color='tomato', ms=7, label='Simulated measurement')
ax.axvline(psutil.cpu_count(logical=False) or 8, color='green', linestyle='--',
           label=f'Physical cores ({psutil.cpu_count(logical=False) or 8})')

ax.set_xlabel('n_threads', fontsize=12)
ax.set_ylabel('Throughput (tok/s)', fontsize=12)
ax.set_title('Scaling with Thread Count — llama.cpp CPU Inference', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks(threads)

save(fig, 'fig12_thread_scaling')

## 15  Save Full JSON Report

In [ ]:
# ── Aggregate all metrics into a single JSON report ────────────────────────
report = {
    'timestamp': TIMESTAMP,
    'miniforge_version': '0.1.0',
    'hardware': {
        'cpu_physical_cores': psutil.cpu_count(logical=False),
        'cpu_logical_cores':  psutil.cpu_count(logical=True),
        'total_ram_gb': round(psutil.virtual_memory().total / (1024 ** 3), 2),
        'available_ram_gb': round(psutil.virtual_memory().available / (1024 ** 3), 2),
    },
    'tests': {
        'passed': passed,
        'failed': failed,
        'errors': errors,
        'skipped': skipped,
    },
    'memory': {
        'max_usable_gb': round(mem.max_usable_gb, 2),
    },
    'throughput_sim': {
        pt: {'mean': round(np.mean(vals), 2), 'std': round(np.std(vals), 2)}
        for pt, vals in samples.items()
    },
    'quality_sim': {
        'Q4_K_M':   dict(zip(categories, scores_q4km)),
        'Q3_K_M':   dict(zip(categories, scores_q3km)),
        'IQ2_XXS':  dict(zip(categories, scores_iq2xxs)),
    },
    'figures_saved': [
        f'fig{n:02d}_*.png' for n in range(1, 13)
    ]
}

report_path = f'{OUTPUT_DIR}/miniforge_report_{TIMESTAMP}.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved → {report_path}')
print(json.dumps(report, indent=2))

## 16  (Optional) Run Real Benchmark Suite

In [ ]:
# Uncomment and set MODEL_PATH to run the actual benchmark suite
# Requires a GGUF model file accessible from Colab (e.g. via Drive or HF)

# MODEL_PATH = '/content/drive/MyDrive/models/minimax-m2.7-IQ2_XXS.gguf'
# HF_TOKEN   = 'hf_...'  # only needed if downloading from a gated repo

# import asyncio
# from benchmarks.benchmark_suite import BenchmarkSuite
# from benchmarks.visualization import BenchmarkVisualizer
# from pathlib import Path
#
# suite = BenchmarkSuite(model_path=MODEL_PATH)
# results = asyncio.run(suite.run_all())
#
# result_path = Path(f'{OUTPUT_DIR}/benchmark_results_{TIMESTAMP}.json')
# with open(result_path, 'w') as fh:
#     import json, dataclasses
#     json.dump(results, fh, default=lambda o: vars(o), indent=2)
#
# viz = BenchmarkVisualizer(result_path)
# md_path = Path(f'{OUTPUT_DIR}/benchmark_report_{TIMESTAMP}.md')
# viz.save_markdown_report(md_path)
# print(f'Benchmark report → {md_path}')

print('Uncomment the block above once you have a GGUF model available.')

---
All figures and the JSON report are in your **Google Drive** at `miniforge_results/`.

To run real benchmarks, mount a GGUF model to Drive and uncomment Section 16.